# 3교시 · ChartQA 전처리 · 타깃 답 설계 · 라벨 마스킹

> **VLM 경량화 2일 과정 · Day 1 (3교시)**
> 실습 환경: **Google Colab (T4 GPU)** · 모델: **Qwen/Qwen3-VL-4B-Instruct** · 데이터: **HuggingFaceM4/ChartQA**

**이 교시의 목표**
- ChartQA 구조(이미지·질문·정답)를 **처음 보는 사람도** 따라오도록 분석한다.
- 모델이 **짧고 일정한 형식**으로 답하도록 **시스템 프롬프트 + 타깃 답**을 설계한다.
- **수업용 서브셋**(train 3K · val 300)을 만들어 **`VLM_FT_2`에 저장**한다(4·5·6교시 재사용).
- VLM 학습의 핵심인 **라벨 마스킹(정답에만 loss)** 콜레이터를 구현한다.

> 🧑‍🏫 이 노트북은 **선수 과정(VLM 파인튜닝)의 전처리 패턴을 그대로 잇습니다** — `shrink` · `to_messages` · 라벨 마스킹 `collate_fn` 구조가 동일합니다. 달라지는 건 *데이터(CORD→ChartQA)* 와 *시스템 프롬프트*뿐입니다. 각 셀 출력 아래 **🔍 결과 해석**을 천천히 보세요.


## 0. 공통 헤더 — Google Drive(VLM_FT_2) 마운트 + HF_TOKEN 로드

> 📌 **모든 Day 1 노트북은 이 셀을 먼저 실행합니다.** Google Drive의 작업 폴더 **`VLM_FT_2`** 를 마운트하고, `.env`의 **HF_TOKEN**을 불러옵니다.
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(전처리 데이터·어댑터·결과가 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다(다운로드 경고 방지·비공개 모델 접근). `login()`은 호출하지 않습니다(환경변수만으로 충분, 경고 방지).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · Google Drive(VLM_FT_2) 마운트 + HF_TOKEN(.env) 로드
#  (모든 Day1 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) Google Drive 마운트 → 작업 폴더 VLM_FT_2 (교시 간 데이터·어댑터·결과 공유)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VLM_DIR = '/content/drive/MyDrive/VLM_FT_2'      # Drive 경로(권장)
except Exception:
    VLM_DIR = '/content/VLM_FT_2'                     # Colab 아니면 로컬 폴백
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 데이터만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /content/drive/MyDrive/VLM_FT_2


## 0. 1·2교시 환경 재설치 (런타임이 끊겼을 수 있으므로)

전처리는 **모델 가중치 없이 프로세서만** 필요합니다. 선수 과정과 동일하게 버전을 고정 설치합니다(`qwen-vl-utils` 포함).

In [2]:
# 2교시와 동일: 추가 패키지만 버전 고정 설치 (torch/torchvision 재설치 금지)
!pip install -q "transformers==5.12.1" "pillow==11.3.0" "datasets>=2.20" "qwen-vl-utils>=0.0.8" "python-dotenv"
# pillow가 12로 보이면(이미 import된 상태) 런타임 다시 시작 후 재실행하세요.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 62.2 MB/s eta 0:00:00:00:0100:01


## 1. 준비 — 프로세서 · 토큰(.env) · 작업 폴더

`AutoProcessor`(토크나이저+이미지 전처리)를 로드하고, 학습용 **오른쪽 패딩**을 지정합니다. 그래야 정답 토큰이 왼쪽에 정렬되고 뒤쪽 패딩만 -100으로 가리기 쉽습니다(선수 과정과 동일).

`.env`의 HF 토큰을 불러와 다운로드 rate-limit 경고를 없애고, `VLM_FT_2` 작업 폴더를 잡습니다.

In [3]:
# 기본 라이브러리
import torch, json, os

# 모델별 입력 전처리기:
# - 텍스트: 토크나이징
# - 이미지: 리사이즈/정규화 등 모델 입력 형태로 변환
from transformers import AutoProcessor

# Qwen VL 메시지 포맷에서 이미지 정보를 추출/정리하는 유틸
from qwen_vl_utils import process_vision_info

# 데이터셋 로드/디스크 저장본 로드를 위한 함수
from datasets import load_dataset, load_from_disk

# 이미지 객체 처리용 PIL
from PIL import Image

# 사용할 베이스 모델 ID
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'

# Processor 로드:
# 전처리 단계에서는 모델 가중치(수 GB)를 올리지 않고 Processor만 있으면 충분
processor = AutoProcessor.from_pretrained(MODEL_ID)

# 학습 시 패딩 방향을 오른쪽으로 고정
# -> 정답 토큰 위치 정렬이 쉬워지고, 뒤쪽 패딩만 -100 마스킹하기 편함
processor.tokenizer.padding_side = 'right'

# VLM_DIR / DATA_DIR / HF_TOKEN은 상단 공통 헤더 셀에서 이미 정의됨
print(
    '프로세서 준비 완료 | padding_side =',
    processor.tokenizer.padding_side,
    '| VLM_DIR =',
    VLM_DIR
)

프로세서 준비 완료 | padding_side = right | VLM_DIR = /content/drive/MyDrive/VLM_FT_2


### 🔍 결과 해석 — 프로세서 준비

- `프로세서 준비 완료 | padding_side = right | VLM_DIR = .../VLM_FT_2` 가 보이면 정상입니다.
- **왜 `padding_side='right'`인가?** 배치 안에서 길이가 다른 문장을 같은 길이로 맞추려고 빈자리를 패딩 토큰으로 채웁니다. 패딩을 **오른쪽(문장 끝)** 에 두면 정답 토큰이 항상 **왼쪽의 같은 구간**에 정렬돼, 뒤쪽 패딩만 -100으로 가리면 됩니다. 왼쪽 패딩이면 정답 위치가 샘플마다 밀려 마스킹이 까다로워집니다.
- 상단 헤더의 `Drive already mounted ...`(이미 마운트됨)·`HF_TOKEN: 로드됨`은 **정상 메시지**입니다. 토큰이 로드되면 데이터 다운로드 시 rate-limit 경고가 사라집니다.

## 1. ChartQA 구조 들여다보기

ChartQA 한 행은 **`image`**(차트), **`query`**(질문), **`label`**(정답 리스트)로 이뤄집니다. 여기선 *차트+질문→짧은 답*입니다.

In [5]:
ds_val = load_dataset('HuggingFaceM4/ChartQA', split='val')   # 최초 1회 다운로드 후 캐시
print('val 개수:', len(ds_val))

# 검증 split의 첫 번째 샘플로 구조 확인
ex = ds_val[0]
print('컬럼:', list(ex.keys()))                                # 데이터셋의 모든 필드명
print('query :', ex['query'])                                 # 질문 텍스트
print('label :', ex['label'], '  ← 리스트(보통 1개)')         # 정답 리스트(str_list, 일반적으로 1개만 사용)
print('image :', ex['image'].size, ex['image'].mode)          # PIL Image 객체의 (너비, 높이) 및 채널 모드(RGB/RGBA 등)

val 개수: 1920
컬럼: ['image', 'query', 'label', 'human_or_machine']
query : What's the color of graph with 56 as the highest value?
label : ['Blue']   ← 리스트(보통 1개)
image : (640, 386) RGB


### 🔍 결과 해석 — ChartQA 구조

- `val 개수: 1920` — 검증 split 전체 크기입니다(곧 300개만 추려 씁니다).
- `컬럼: ['image', 'query', 'label', 'human_or_machine']` — 우리가 쓸 건 **image(차트)·query(질문)·label(정답)** 세 개입니다. `human_or_machine`은 *그 질문이 사람이 만든 것인지/자동 생성인지* 표시하는 메타데이터라 학습엔 쓰지 않습니다.
- 예시 `query: "What's the color of graph with 56 as the highest value?"` / `label: ['Blue']` — **차트를 직접 봐야 풀리는** 질문이고, 정답은 리스트라 항상 `label[0]`(`'Blue'`)로 꺼냅니다.
- `image: (640, 386) RGB` — 이 샘플은 작고 RGB지만 **데이터마다 크기·채널이 제각각**입니다(아래 3절에서 표준화). 정답이 *색·숫자·Yes/No*처럼 **짧다는 점**이 5교시 Relaxed Accuracy와 잘 맞고, 시스템 프롬프트로 *짧게 답하기*를 가르칠 근거가 됩니다.

## 2. 수업용 서브셋 + `VLM_FT_2` 저장

시드를 고정해 섞은 뒤 앞에서 일부만 골라 **누구나 같은 서브셋**을 재현합니다. 만든 서브셋은 **`VLM_FT_2/data`에 저장**해 4·5·6교시가 그대로 불러 씁니다.

In [6]:
# 재현 가능한 셔플을 위한 시드 고정
SEED = 42

# 수업용 서브셋 크기 설정
TRAIN_N, VAL_N = 3000, 300

# train split 로드 → 시드 기반 셔플 → 앞에서 3000개 선택
train_set = load_dataset('HuggingFaceM4/ChartQA', split='train').shuffle(seed=SEED).select(range(TRAIN_N))

# val split은 이미 ds_val로 로드되어 있으므로 재사용 → 시드 셔플 → 앞에서 300개 선택
val_set = ds_val.shuffle(seed=SEED).select(range(VAL_N))

# 서브셋 크기와 첫 샘플 정답 확인
print('train:', len(train_set), '| val:', len(val_set))
print('예시 정답:', train_set[0]['label'][0])

# Drive 작업 폴더(VLM_FT_2/data) 저장 경로 구성
# 저장 형식: Hugging Face Dataset의 Arrow 디스크 포맷
TRAIN_PATH = f'{DATA_DIR}/chartqa_train_{TRAIN_N}'
VAL_PATH   = f'{DATA_DIR}/chartqa_val_{VAL_N}'

# 서브셋 저장 (다음 교시에서 load_from_disk로 즉시 재사용)
train_set.save_to_disk(TRAIN_PATH)
val_set.save_to_disk(VAL_PATH)

# 저장 완료 경로 출력
print('저장 완료 →', TRAIN_PATH, '/', VAL_PATH)

train: 3000 | val: 300
예시 정답: 36


Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

저장 완료 → /content/drive/MyDrive/VLM_FT_2/data/chartqa_train_3000 / /content/drive/MyDrive/VLM_FT_2/data/chartqa_val_300


### 🔍 결과 해석 — 서브셋 / 저장

- `train: 3000 | val: 300` — 의도한 크기입니다. **시드 42**가 같으면 누가 실행해도 **동일한 서브셋**이라 결과를 재현할 수 있습니다.
- `예시 정답: 36` — 학습 타깃이 이렇게 **짧은 한 토막**(숫자·단어)입니다. 모델이 장황하게 설명하지 않고 이 한 토막만 내도록 가르치는 게 목표입니다.
- `Saving the dataset (0/1 shards) ...` 진행바는 정상이며, `저장 완료 → .../VLM_FT_2/data/chartqa_train_3000` 이 뜨면 **Drive에 영구 저장**된 것입니다. 4·5·6교시가 `load_from_disk`로 이 데이터를 그대로 불러 써서 **교시 간 데이터가 100% 일치**합니다(런타임이 끊겨도 재다운로드 불필요).

## 3. 이미지 표준화 (T4 메모리 보호)

이미지가 클수록 패치→토큰이 폭증해 T4를 잡아먹습니다. 선수 과정과 **동일한 `shrink`** 로 긴 변을 768로 제한하고 RGB로 통일합니다. ChartQA 이미지는 원래 작아 대부분 그대로지만, *데이터가 바뀌어도 안전*합니다. (실제 적용은 아래 `collate_fn` 안에서 배치마다 수행)

In [7]:
def shrink(img, max_side=768):
    """
    이미지 전처리 함수
    - 긴 변이 max_side를 넘으면 비율을 유지한 채 축소
    - 최종 채널 모드를 RGB로 통일
    """
    # 원본 이미지의 너비(w), 높이(h)
    w, h = img.size

    # 긴 변 기준으로 축소 비율 계산 (긴 변이 max_side가 되도록)
    s = max_side / max(w, h)

    # 원본이 max_side보다 큰 경우에만 축소 수행
    if s < 1:
        # LANCZOS: 축소 시 품질이 좋은 리샘플링 방식
        img = img.resize((int(w * s), int(h * s)), Image.LANCZOS)

    # 모델 입력 일관성을 위해 RGB로 변환 후 반환
    return img.convert('RGB')

# 앞의 3개 샘플에 대해 전처리 전/후 크기와 모드 확인
for i in range(3):
    im = train_set[i]['image']  # ChartQA 샘플 이미지(PIL)
    print(f'{i}: 원본 {im.size} {im.mode} → shrink {shrink(im).size} {shrink(im).mode}')

0: 원본 (800, 557) RGBA → shrink (768, 534) RGB
1: 원본 (850, 600) RGBA → shrink (768, 542) RGB
2: 원본 (800, 557) RGBA → shrink (768, 534) RGB


### 🔍 결과 해석 — 이미지 표준화

출력을 보면 `shrink`가 실제로 **두 가지** 일을 합니다.
- **크기 축소**: `원본 (800, 557) → shrink (768, 534)` — 긴 변(800)이 상한 768을 넘어, 비율을 유지한 채 768로 줄었습니다(세로도 557→534로 함께). 큰 차트일수록 이미지 토큰이 **제곱으로** 늘어 T4 메모리를 위협하므로, 이 상한이 안전장치입니다.
- **채널 통일(RGBA → RGB)**: 원본이 **RGBA**(빨강·초록·파랑 + **투명도 알파** = 4채널)였는데 **RGB**(3채널)로 바꿨습니다. 모델 비전 타워는 3채널 입력을 기대하므로, 알파 채널이 섞이면 오류·왜곡이 납니다.

> 💡 1절의 val 샘플은 `(640, 386) RGB`라 그대로였지만, 여기 train 샘플들은 `(800, 557) RGBA`라 **크기·채널 둘 다 실제로 변환**됐습니다. 데이터가 섞여 있어도 `shrink` 한 줄이 모두 표준화해 줍니다.

## 4. ⭐ 핵심: 시스템 프롬프트 + 채팅 템플릿 + 라벨 마스킹 콜레이터

이 노트북에서 **가장 중요한** 부분이며, 선수 과정과 **같은 구현**입니다.

**(1) 시스템 프롬프트** — 2교시 베이스라인에서 모델이 `'Blue'` 대신 문단으로 장황하게 답한 걸 봤습니다. 그래서 **"설명 없이 한 단어/숫자로만 답하라"** 는 시스템 메시지로 *형식을 강제*합니다.

**(2) 채팅 템플릿** — `apply_chat_template`이 우리 메시지(역할+내용)를 모델이 학습한 특수토큰 형식 문자열로 바꿔줍니다. 개념 예시:
```text
<|im_start|>system
당신은 차트를 읽고 ... 짧게 답하세요.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>What's the color ...<|im_end|>
<|im_start|>assistant
```

**(3) 라벨 마스킹** — 손실을 **정답 토큰에만** 줍니다. 정답이 아닌 부분(system·user·이미지·패딩)은 **-100**(PyTorch가 무시하기로 약속된 값)으로 가립니다. 구현은 *완성문장*과 *정답 직전까지의 프롬프트* 길이(`plen`)를 비교해 앞부분을 -100으로 덮는 방식(선수 과정과 동일).

> 💡 이 방식(`processor(text=, images=, padding=True)`)은 Qwen3-VL이 요구하는 **`mm_token_type_ids`도 프로세서가 자동 반환·패딩**합니다 — 그래서 별도 처리가 필요 없고 멀티모달 위치 인코딩 오류도 나지 않습니다.

- <|vision_start|><|image_pad|><|vision_end|> 이부분이 LLM에서와 차이점  (비전토큰을 머저가 사이즈 맞춰 제공)

- https://developers.openai.com/api/docs/models/gpt-5.5

1,050,000 context window
128,000 max output tokens
차이가 vision data input으로 어마어마하게 넣을 수 있다

Input
Text, image  

- 비교 : GPT3.5 tubo  16,385 context window



In [9]:
# 시스템 프롬프트:
# - 모델이 장황하게 설명하지 않도록
# - "한 단어/숫자" 형태의 짧은 답변을 강제
SYSTEM = "당신은 차트를 읽고 질문에 답하는 도우미입니다. 설명 없이 한 단어 또는 숫자로만 짧게 답하세요."

def to_messages(image, question, answer=None): #학습할때 정답 주고 추론할때 None
    """
    ChartQA 1개 샘플을 Qwen 채팅 메시지 포맷으로 변환.
    반환 형태:
      - system: 답변 스타일 규칙
      - user: 이미지 + 질문
      - assistant: 정답(학습 시에만 포함)
    """
    msgs = [
        # 모델의 답변 형식을 고정하는 시스템 지시
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM}]},

        # 사용자 입력: 이미지 1장 + 질문 텍스트
        {'role': 'user',   'content': [{'type': 'image', 'image': image},
                                        {'type': 'text',  'text': question}]},
    ]

    # answer가 있을 때만 assistant 턴 추가(학습용 타깃 생성)
    # answer가 없으면 추론 프롬프트(정답 직전) 용도로 사용
    if answer is not None:
        msgs.append({'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]})
    return msgs

def collate_fn(batch):
    """
    DataLoader용 collate 함수.
    입력: batch = [{image, query, label}, ...]
    출력: model_inputs(dict) + labels(정답 토큰만 loss가 걸리도록 -100 마스킹)
    """

    # 1) 이미지 전처리(긴 변 제한 + RGB 통일)
    imgs = [shrink(b['image']) for b in batch]

    # 2) 메시지 2종 생성
    # - full_msgs  : 정답까지 포함된 완성 대화(실제 입력 시퀀스)
    # - prompt_msgs: 정답 없는 프롬프트(정답 시작 위치 계산용)
    full_msgs   = [to_messages(im, b['query'], str(b['label'][0])) for im, b in zip(imgs, batch)]
    prompt_msgs = [to_messages(im, b['query'], None)               for im, b in zip(imgs, batch)]

    # 3) 채팅 템플릿 문자열로 변환
    # - add_generation_prompt=False: assistant 답변까지 포함된 전체 텍스트
    # - add_generation_prompt=True : assistant 헤더에서 끝나는 프롬프트 텍스트
    # tokenize=True면 token id리턴  Fasle면 text 리턴
    full_texts   = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in full_msgs]
    prompt_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)  for m in prompt_msgs]

    # 4) 비전 입력 추출(샘플당 이미지 1장)
    image_inputs = []
    for m in full_msgs:
        ii, _ = process_vision_info(m)
        image_inputs.extend(ii)

    # 5) 멀티모달 일괄 토크나이즈 + 패딩
    # 반환 예: input_ids, attention_mask, pixel_values, mm_token_type_ids
    model_inputs = processor(text=full_texts, images=image_inputs, padding=True, return_tensors='pt')

    # 6) 라벨 생성: input_ids 복사 후, loss 제외 구간을 -100 처리
    labels = model_inputs['input_ids'].clone()

    # (a) 패딩 토큰은 loss 계산에서 제외
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # (b) 각 샘플에서 "정답 시작 전" 구간(system/user/이미지/assistant 헤더) 제외
    for i in range(len(batch)):
        # prompt 길이(plen) = 정답 직전까지의 토큰 길이
        plen = processor(
            text=[prompt_texts[i]],
            images=[image_inputs[i]],
            return_tensors='pt'
        )['input_ids'].shape[1]

        # 앞부분을 전부 -100으로 마스킹 → 정답 토큰에만 loss 적용
        labels[i, :plen] = -100

    # 7) model_inputs에 labels 추가
    model_inputs['labels'] = labels
    return model_inputs

print('collate_fn 준비 완료 (시스템 프롬프트 + 라벨 마스킹)')

collate_fn 준비 완료 (시스템 프롬프트 + 라벨 마스킹)


## 4-B. 개념 — `input_ids` · `attention_mask` · `pixel_values` · `mm_token_type_ids` 의 관계

위 `collate_fn`의 `processor(text=…, images=…, padding=True)` **한 줄**이 이 네 텐서를 한꺼번에 만들어 반환합니다. 역할과 **모양(shape)** 으로 나눠 보면 관계가 분명해집니다.

**① 세 텐서는 같은 shape `(B, L)`** — 토큰 위치마다 1:1로 겹칩니다.
- `input_ids` — 토큰 ID 시퀀스. VLM에선 이미지가 `` `<|vision_start|>` `` + `` `<|image_pad|>` `` ×N + `` `<|vision_end|>` `` 로 **여러 개의 이미지 토큰**으로 펼쳐져 들어갑니다. (그래서 아래 배치가 길이 474처럼 길어지고, 그 **대부분이 이미지 토큰**입니다.)
- `attention_mask` — 같은 자리에 `1`(실제 토큰)/`0`(패딩). 모델이 패딩을 무시하게 합니다.
- `mm_token_type_ids` — 같은 자리에 그 토큰이 **텍스트(0)인지 이미지(1)인지** 표시.

**② 나머지 하나 `pixel_values` 는 shape가 다릅니다.** 시퀀스 축이 아니라 **이미지 패치 축**을 가진 별도 텐서로, 이미지의 *실제 픽셀 내용*을 담습니다. (함께 나오는 `image_grid_thw` 는 이미지별 `(t, h, w)` 패치 격자입니다.)

**한 줄로 정렬해 보기** — `User [이미지] 차트?` + 패딩 1칸, 이미지가 토큰 3개로 펼쳐졌다고 가정(`<vs>`=vision_start, `<ve>`=vision_end, IMG=image_pad):

| 위치 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| `input_ids` | User | `<vs>` | IMG | IMG | IMG | `<ve>` | 차트? | PAD |
| `attention_mask` | 1 | 1 | 1 | 1 | 1 | 1 | 1 | **0** |
| `mm_token_type_ids` | 0 | 0 | **1** | **1** | **1** | 0 | 0 | 0 |

`pixel_values` 는 이 표 안에 없습니다 — 대신 **위치 2·3·4(이미지 토큰)** 자리에 끼워 넣을 "재료"를 따로 담고 있습니다.
머저를 통해 들어오는 원본

**순전파에서 어떻게 만나나**
1. `input_ids` 의 텍스트 토큰 → 임베딩 테이블로 벡터화
2. `pixel_values` → **비전 인코더(ViT+merger)** → 이미지 임베딩 N개 생성
3. 그 N개를 **`mm_token_type_ids==1`(= `input_ids` 가 `<|image_pad|>`) 위치에 끼워 넣어** 시퀀스를 완성
4. **M-RoPE**(Qwen3-VL의 위치 인코딩)가 `mm_token_type_ids` · `image_grid_thw` 를 보고, 텍스트엔 1D 위치를, 이미지 토큰엔 2D/3D `(t,h,w)` 좌표를 부여

> **불변식**: `mm_token_type_ids==1` 인 칸 수 = `pixel_values`/`grid_thw` 가 만드는 (머지 후) 패치 수. 둘이 안 맞으면 *"image features와 image tokens 수 불일치"* 에러가 납니다. 프로세서는 이 둘을 **항상 맞춰서** 내보냅니다.

**왜 배치에서 `mm_token_type_ids` 가 중요한가** — 길이가 다른 샘플을 `padding=True` 로 묶으면 이미지 토큰의 **위치가 샘플마다 밀립니다.** `mm_token_type_ids` 는 `input_ids` 와 **함께 패딩**되어 따라다니므로, 패딩 후에도 모델이 "어디가 이미지였는지"를 잃지 않습니다. 그래서 직접 손으로 텐서를 만들지 않고 **`processor(text=…, images=…, padding=True)` 한 번으로 네 개를 같이 받아 같이 패딩**하는 이 방식이 안전합니다. (Qwen3-VL은 `mm_token_type_ids` 를 프로세서가 자동 반환·패딩 — 그래서 **아래 관찰 셀에서 `mm_token_type_ids? : True`가 떠야 정상**입니다.)

마지막으로 `labels`(위 6번에서 생성)는 `input_ids` 를 복제해 **같은 `(B, L)`** 로 만든 뒤 패딩·프롬프트 구간을 `-100` 으로 가린 것이라, 위 세 텐서와 **같은 정렬축**을 공유합니다 — 그래서 라벨 마스킹이 위치 단위로 정확히 맞아떨어집니다.

**한눈에**: `input_ids`=무엇을 · `attention_mask`=어디까지 볼지 · `mm_token_type_ids`=어디가 이미지인지 · `pixel_values`=이미지의 실제 내용. **앞의 셋은 시퀀스 위에 겹쳐 정렬**되고, **`pixel_values` 는 이미지 토큰 위치를 통해** 그 시퀀스에 연결됩니다.

## 5. 🔍 관찰 포인트 — 라벨이 '정답만' 남았는지 눈으로 확인

배치 1개를 만들어 **라벨에서 -100이 아닌(=학습 대상) 토큰만** 디코딩합니다. 그 결과가 **정답과 일치**하면 마스킹 성공입니다.

In [10]:
# collate_fn으로 2개 샘플을 묶어 미니 배치를 생성
mini = collate_fn([train_set[0], train_set[1]])

# 입력 텐서의 크기 확인
print('input_ids shape   :', tuple(mini['input_ids'].shape))   # (배치 크기, 시퀀스 길이)

# 이미지 입력이 포함되었는지 확인
print('pixel_values 존재? :', 'pixel_values' in mini)

# 멀티모달 위치 정보(mm_token_type_ids)가 포함되었는지 확인
print('mm_token_type_ids? :', 'mm_token_type_ids' in mini)     # 자동 포함되어야 정상
print('-' * 60)

# 각 샘플별로 라벨에서 -100이 아닌 토큰만 꺼내어 확인
for i, lab in enumerate(mini['labels']):
    keep = lab[lab != -100]                                    # 학습 대상 토큰만 추출
    print(f'[샘플{i}] 학습 대상 토큰 수:', keep.numel())       # 실제 loss가 걸리는 토큰 개수
    print(f'[샘플{i}] 학습 대상 디코딩 :', repr(processor.tokenizer.decode(keep)))  # 디코딩 결과
    print(f'[샘플{i}] 실제 정답       :', repr(str(train_set[i]['label'][0])))     # 원본 정답

input_ids shape   : (2, 474)
pixel_values 존재? : True
mm_token_type_ids? : True
------------------------------------------------------------
[샘플0] 학습 대상 토큰 수: 4
[샘플0] 학습 대상 디코딩 : '36<|im_end|>\n'
[샘플0] 실제 정답       : '36'
[샘플1] 학습 대상 토큰 수: 3
[샘플1] 학습 대상 디코딩 : 'Yes<|im_end|>\n'
[샘플1] 실제 정답       : 'Yes'


### 🔍 결과 해석 — 라벨 마스킹이 제대로 됐는가 (가장 중요)

- `input_ids shape: (2, 474)` → 배치 2개를 **길이 474**로 맞췄습니다. 474 중 **대부분이 이미지 토큰**이고, 질문·정답 텍스트는 몇 개뿐입니다. `pixel_values: True`, **`mm_token_type_ids: True`** → 이미지 픽셀과 멀티모달 위치 정보까지 모두 담겼습니다.
- `[샘플0] 학습 대상 토큰 수: 4` → 474개 중 **단 4개에만** loss가 걸립니다. 나머지 470개(이미지+질문+패딩)는 -100으로 가려졌습니다. 이것이 라벨 마스킹의 핵심 — *"정답 부분만 채점"*.
- **디코딩이 정답과 일치**: 샘플0은 `'36...'` ↔ 정답 `'36'`, 샘플1은 `'Yes...'` ↔ `'Yes'`. ✅ 정답 토큰만 정확히 남았습니다.
- **왜 답 뒤에 `<|im_end|>`(종료 토큰)까지 학습 대상인가?** — 디코딩 결과가 `'36'`이 아니라 `'36<|im_end|>'`(+줄바꿈)인 데 주목하세요. 학습 타깃에 **답 다음의 종료 토큰**이 포함돼야 모델이 *"짧게 답하고 여기서 멈춰라"* 를 배웁니다. 2교시 베이스라인이 장황했던 건 바로 이 **멈추는 법**을 안 배웠기 때문이고, 시스템 프롬프트 + 이 종료 토큰 학습이 그걸 교정합니다. (그래서 샘플0=4개, 샘플1=3개처럼 *정답 길이에 종료 토큰 1~2개를 더한* 만큼만 학습 대상이 됩니다.)
- 만약 디코딩에 **질문 문장이나 이미지 토큰**이 섞여 보이면 마스킹이 잘못된 것입니다 → `padding_side='right'`와 `plen`(프롬프트 길이) 계산을 다시 점검하세요.

## 6. 정리 & 다음 교시

- ChartQA를 분석하고 **시스템 프롬프트**(짧게 답하기)와 **라벨 마스킹 `collate_fn`** 을 *선수 과정과 같은 패턴*으로 구현했습니다.
- 서브셋을 **`VLM_FT_2/data`에 저장**해 4·5·6교시가 재사용합니다.

### 다음 교시 — Day1-4 · QLoRA 학습
4bit 동결 base + **LoRA 어댑터**를 얹어 **T4에서 파인튜닝**합니다. 4교시는 여기서 저장한 서브셋과 **동일한 `collate_fn`**(시스템 프롬프트 포함)을 그대로 씁니다.

> ✅ **체크포인트**: ① 시스템 프롬프트가 왜 필요한지 안다(베이스라인 장황함) ② 학습 대상 디코딩 = 정답을 확인했다 ③ `mm_token_type_ids`가 자동 포함됨을 확인했다 ④ 서브셋이 `VLM_FT_2/data`에 저장됐다.